# Лабораторная работа №9. IPv6 и Neighbor Discovery

IPv6 — это не просто более длинный адрес. Посмотрим link-local/global адреса, `/64`, multicast и Neighbor Discovery вместо ARP.

## Схема

![IPv6](assets/ipv6.svg)

Два Linux-хоста соединены напрямую по `eth1`. Сначала изучим уже существующий link-local, затем добавим documentation-prefix `2001:db8:9::/64`.

## 1. Что есть до нашей настройки

На `pc1`:

```bash
ip -6 addr show dev eth1
ip -6 route
```

Даже без global address Linux обычно создаёт link-local `fe80::/64`. Он имеет link scope и не предназначен для обычной глобальной маршрутизации.

### Q1. Почему link-local адрес полезен, даже если global IPv6 ещё не настроен?

`введите ваш ответ`

## 2. Добавим global addresses

`pc1`:

```bash
ip -6 addr add 2001:db8:9::10/64 dev eth1
```

`pc2`:

```bash
ip -6 addr add 2001:db8:9::20/64 dev eth1
```

На `pc1`:

```bash
ip -6 route get 2001:db8:9::20
ip -6 neigh flush dev eth1
```

### Q2. Почему `/64` в IPv6 встречается настолько часто на обычном LAN и какую роль он играет для Neighbor Discovery/SLAAC?

`введите ваш ответ`

In [ ]:
%%capture_traffic pc1:eth1 --filter "icmpv6" --timeout 12 --packets 100 --save captures/ndp.pcap
import time
print("На pc1: ping -6 -c 2 2001:db8:9::20")
time.sleep(10)

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
viewer = PcapViewer("captures/ndp.pcap", "icmpv6", limit=40)
frames = viewer.packets()
display(frames)
if not frames.empty:
    for n in frames["№"].head(5):
        print("\nframe", n)
        print("\n".join(viewer.packet(int(n), "protocols").splitlines()[:70]))

## 3. Neighbor Discovery вместо ARP

После ping:

```bash
ip -6 neigh show dev eth1
```

Ищите Neighbor Solicitation и Neighbor Advertisement. Solicitation обычно идёт не на Ethernet broadcast, а на solicited-node multicast.

### Q3. Чем multicast Neighbor Solicitation принципиально отличается от IPv4 ARP broadcast?

`введите ваш ответ`

## CHECK

Checker проверяет global IPv6 `/64` на обоих хостах и реальный `ping -6` между ними.

## Итог

IPv6 neighbor discovery встроен в ICMPv6 и активно использует multicast. Link-local — базовая часть работы IPv6 link, а global prefix добавляет адресуемость выше link scope.